# Notebook 03: Structured Experimental Design (Part III)

**Objective:** Conduct at least ONE ROUND of controlled experiments, varying a single hyperparameter at a time.

## Experimental Plan

All experiments use the same baseline training protocol (yolo11s, 50 epochs).
Each experiment changes **one variable** at a time.

| Experiment | Variable Changed | Values |
|---|---|---|
| Baseline |  -  | imgsz=640, model=yolo11s |
| E1 | Image resolution | 1280 |
| E2 | Model size | yolo11n, yolo11m |
| E3 | Batch size | 8, 32 |
| E4 | Learning rate | 0.001, 0.1 |


In [1]:
import sys
from pathlib import Path
import os
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from ultralytics import YOLO
from ultralytics.utils.tal import TaskAlignedAssigner

# --- MPS workaround ---
# Store the true original _forward on the class the first time this runs.
# Subsequent re-runs of this cell will skip the save but still re-apply the patch,
# and the patch always calls _original_forward (the real original), so no recursion.
if not hasattr(TaskAlignedAssigner, '_original_forward'):
    TaskAlignedAssigner._original_forward = TaskAlignedAssigner._forward

def _tal_forward_mps_safe(self, pd_scores, pd_bboxes, anc_points,
                           gt_labels, gt_bboxes, mask_gt):
    dev = pd_scores.device
    if dev.type == 'mps':
        result = TaskAlignedAssigner._original_forward(
            self,
            pd_scores.cpu(), pd_bboxes.cpu(), anc_points.cpu(),
            gt_labels.cpu(), gt_bboxes.cpu(), mask_gt.cpu(),
        )
        return tuple(t.to(dev) if isinstance(t, torch.Tensor) else t for t in result)
    return TaskAlignedAssigner._original_forward(
        self, pd_scores, pd_bboxes, anc_points, gt_labels, gt_bboxes, mask_gt)

TaskAlignedAssigner._forward = _tal_forward_mps_safe
# --- end workaround ---

sns.set_theme(style='whitegrid')

DATA_CFG    = ROOT / 'configs' / 'visdrone.yaml'
PROJECT_DIR = ROOT / 'results'
DEVICE      = 'mps'    # Apple Silicon GPU; MPS TAL bug is patched above
WORKERS     = 0        # ultralytics overrides to 0 on MPS anyway; explicit is cleaner

# Shared baseline parameters for ablation experiments.
# Epochs=25 and patience=10 are enough to compare relative differences;
# full 50-epoch runs are reserved for the final best config in notebook 01.
# cache is intentionally OFF  -  SSD reads at 5 GB/s and RAM cache at large
# image sizes consumes tens of GB, risking system instability.
BASE_CONFIG = dict(
    data     = str(DATA_CFG),
    epochs   = 25,
    batch    = 16,
    lr0      = 0.01,
    lrf      = 0.01,
    patience = 10,
    device   = DEVICE,
    workers  = WORKERS,
    project  = str(PROJECT_DIR / 'experiments'),
    exist_ok = True,
    plots    = True,
    verbose  = False,
)

print('Setup complete. DATA_CFG:', DATA_CFG)
print('PyTorch:', torch.__version__, '| MPS available:', torch.backends.mps.is_available())


Setup complete. DATA_CFG: /Users/toriav/Desktop/Erem/CMPE 401/Instructor Projects/Project 1/Object-Detection-Study-using-YOLOv11-or-YOLOv26-/configs/visdrone.yaml
PyTorch: 2.11.0 | MPS available: True


## Helper: Train + Evaluate

In [2]:
def run_experiment(checkpoint: str, exp_name: str, override: dict) -> dict:
    """Train a model with one config override and return metrics.

    If the experiment already completed (metrics.json exists in its save
    directory), the saved result is returned immediately without re-training.
    This makes the notebook safe to stop and restart at any point.
    """
    # --- Resume from cache ---
    expected_save_dir = Path(BASE_CONFIG['project']) / exp_name
    cached_path = expected_save_dir / 'metrics.json'
    if cached_path.exists():
        with open(cached_path) as f:
            metrics = json.load(f)
        print(f"\n[CACHED] {exp_name}  -  skipping re-train, loaded saved result.")
        print(f"  mAP50={metrics['mAP50']:.4f}  mAP95={metrics['mAP50_95']:.4f}")
        return metrics

    # --- Fresh training run ---
    cfg = {**BASE_CONFIG, 'name': exp_name, **override}

    print(f"\n{'-'*55}")
    print(f"  Running: {exp_name}")
    for k, v in override.items():
        print(f"    {k} = {v}")
    print(f"{'-'*55}")

    model = YOLO(checkpoint)
    t0 = time.time()
    results = model.train(**cfg)
    elapsed = time.time() - t0

    rd = results.results_dict
    metrics = {
        'experiment':  exp_name,
        'model':       checkpoint,
        'override':    override,
        'mAP50':       round(float(rd.get('metrics/mAP50(B)', 0)), 4),
        'mAP50_95':    round(float(rd.get('metrics/mAP50-95(B)', 0)), 4),
        'precision':   round(float(rd.get('metrics/precision(B)', 0)), 4),
        'recall':      round(float(rd.get('metrics/recall(B)', 0)), 4),
        'train_min':   round(elapsed / 60, 1),
        'save_dir':    str(results.save_dir),
    }

    out_path = Path(results.save_dir) / 'metrics.json'
    with open(out_path, 'w') as f:
        json.dump(metrics, f, indent=2)

    print(f"  mAP50={metrics['mAP50']:.4f}  mAP95={metrics['mAP50_95']:.4f}  time={elapsed/60:.1f}min")
    return metrics

In [3]:
# -- Restore session ------------------------------------------------------------
# Run this cell after a kernel restart to reload any experiments that already
# finished. Completed experiments will not be re-trained when their cells run.
def _load_cached(exp_name):
    p = Path(BASE_CONFIG['project']) / exp_name / 'metrics.json'
    if p.exists():
        with open(p) as f:
            m = json.load(f)
        print(f"  [restored] {exp_name}: mAP50={m['mAP50']:.4f}")
        return m
    return None

e1_metrics  = _load_cached('e1_imgsz1280')
e2n_metrics = _load_cached('e2_model_nano')
e2m_metrics = _load_cached('e2_model_medium')
e3_low_lr   = _load_cached('e3_lr_0001')
e3_cos_lr   = _load_cached('e3_cos_lr')

_n = sum(1 for m in [e1_metrics, e2n_metrics, e2m_metrics, e3_low_lr, e3_cos_lr] if m is not None)
print(f"\n{_n}/5 experiments restored from disk.")


0/5 experiments restored from disk.


## Experiment E1  -  Image Resolution (640 vs 1280)

**Hypothesis**: Higher input resolution should help detect the small, dense objects in VisDrone.  
**Expected outcome**: Improved mAP, especially for small-object classes, at the cost of training time.

In [4]:
# Note: baseline (imgsz=640) already run in Notebook 01
# Load its metrics
baseline_json = ROOT / 'results' / 'baseline' / 'baseline_metrics.json'
if baseline_json.exists():
    with open(baseline_json) as f:
        b = json.load(f)
    baseline_row = {
        'experiment': 'baseline_640',
        'model': b.get('model', 'yolo11s.pt'),
        'override': {'imgsz': 640},
        'mAP50': b['mAP50'],
        'mAP50_95': b['mAP50_95'],
        'precision': b['precision'],
        'recall': b['recall'],
        'train_min': 0,  # already done
    }
    print('Baseline loaded:', baseline_row)
else:
    print('Baseline metrics not found  -  run Notebook 01 first!')
    baseline_row = None

Baseline loaded: {'experiment': 'baseline_640', 'model': 'yolo11s.pt', 'override': {'imgsz': 640}, 'mAP50': 0.3758, 'mAP50_95': 0.2176, 'precision': 0.5058, 'recall': 0.3892, 'train_min': 0}


In [5]:
# E1: Higher resolution
# NOTE: imgsz=1280 requires more VRAM  -  reduce batch if OOM
e1_metrics = run_experiment(
    checkpoint='yolo11s.pt',
    exp_name='e1_imgsz1280',
    override={'imgsz': 1280, 'batch': 8},   # smaller batch for larger images
)


───────────────────────────────────────────────────────
  Running: e1_imgsz1280
    imgsz = 1280
    batch = 8
───────────────────────────────────────────────────────
Ultralytics 8.4.37 🚀 Python-3.11.14 torch-2.11.0 MPS (Apple M4 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/toriav/Desktop/Erem/CMPE 401/Instructor Projects/Project 1/Object-Detection-Study-using-YOLOv11-or-YOLOv26-/configs/visdrone.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1280, int8=False, iou=0.7, keras=False, kobj=1.0,

## Experiment E2  -  Model Size (nano vs small vs medium)

**Hypothesis**: Larger model capacity should improve mAP but risk overfitting on the ~6k training images.  
**Expected outcome**: yolo11m > yolo11s > yolo11n on mAP; yolo11m may show more val–train gap.

In [6]:
e2n_metrics = run_experiment(
    checkpoint='yolo11n.pt',
    exp_name='e2_model_nano',
    override={'imgsz': 640},
)

e2m_metrics = run_experiment(
    checkpoint='yolo11m.pt',
    exp_name='e2_model_medium',
    override={'imgsz': 640, 'batch': 8},
)


───────────────────────────────────────────────────────
  Running: e2_model_nano
    imgsz = 640
───────────────────────────────────────────────────────
Ultralytics 8.4.37 🚀 Python-3.11.14 torch-2.11.0 MPS (Apple M4 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/toriav/Desktop/Erem/CMPE 401/Instructor Projects/Project 1/Object-Detection-Study-using-YOLOv11-or-YOLOv26-/configs/visdrone.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=No

## Experiment E3  -  Learning Rate

**Hypothesis**: A lower LR may give more stable convergence; too high leads to instability.  
**Expected outcome**: LR=0.001 may converge more slowly but reach better final mAP.

In [7]:
e3_low_lr = run_experiment(
    checkpoint='yolo11s.pt',
    exp_name='e3_lr_0001',
    override={'imgsz': 640, 'lr0': 0.001, 'lrf': 0.01},
)

e3_cos_lr = run_experiment(
    checkpoint='yolo11s.pt',
    exp_name='e3_cos_lr',
    override={'imgsz': 640, 'cos_lr': True},
)


───────────────────────────────────────────────────────
  Running: e3_lr_0001
    imgsz = 640
    lr0 = 0.001
    lrf = 0.01
───────────────────────────────────────────────────────
Ultralytics 8.4.37 🚀 Python-3.11.14 torch-2.11.0 MPS (Apple M4 Max)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/Users/toriav/Desktop/Erem/CMPE 401/Instructor Projects/Project 1/Object-Detection-Study-using-YOLOv11-or-YOLOv26-/configs/visdrone.yaml, degrees=0.0, deterministic=True, device=mps, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=25, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=Fa

## Results Table  -  All Experiments

In [8]:
all_results = [m for m in [
    baseline_row,
    e1_metrics,
    e2n_metrics,
    e2m_metrics,
    e3_low_lr,
    e3_cos_lr,
] if m is not None]

results_df = pd.DataFrame(all_results)[[
    'experiment', 'mAP50', 'mAP50_95', 'precision', 'recall', 'train_min'
]].set_index('experiment')

print('\nExperimental Results Table')
print('=' * 65)
print(results_df.to_string())

results_df.to_csv(ROOT / 'results' / 'experiments' / 'experiment_results.csv')
print('\nSaved to results/experiments/experiment_results.csv')


Experimental Results Table
                  mAP50  mAP50_95  precision  recall  train_min
experiment                                                     
baseline_640     0.3758    0.2176     0.5058  0.3892        0.0
e1_imgsz1280     0.1170    0.0698     0.3170  0.1537      246.4
e2_model_nano    0.1588    0.0890     0.3346  0.2005      165.3
e2_model_medium  0.4197    0.2521     0.5460  0.4236      297.3
e3_lr_0001       0.3601    0.2101     0.4913  0.3697      155.9
e3_cos_lr        0.3652    0.2133     0.4963  0.3722      141.2

Saved to results/experiments/experiment_results.csv


## Visualisation  -  Experiment Comparison

In [9]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

x = range(len(results_df))
colors = plt.cm.tab10.colors

# mAP50
bars = axes[0].bar(x, results_df['mAP50'], color=colors[:len(results_df)])
axes[0].set_xticks(x)
axes[0].set_xticklabels(results_df.index, rotation=30, ha='right', fontsize=8)
axes[0].set_ylabel('mAP@50')
axes[0].set_title('mAP@50 by Experiment')
for bar, v in zip(bars, results_df['mAP50']):
    axes[0].text(bar.get_x() + bar.get_width()/2, v + 0.002, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=7)

# mAP50-95
bars = axes[1].bar(x, results_df['mAP50_95'], color=colors[:len(results_df)])
axes[1].set_xticks(x)
axes[1].set_xticklabels(results_df.index, rotation=30, ha='right', fontsize=8)
axes[1].set_ylabel('mAP@50-95')
axes[1].set_title('mAP@50-95 by Experiment')
for bar, v in zip(bars, results_df['mAP50_95']):
    axes[1].text(bar.get_x() + bar.get_width()/2, v + 0.002, f'{v:.3f}',
                 ha='center', va='bottom', fontsize=7)

# Precision vs Recall
axes[2].scatter(results_df['recall'], results_df['precision'],
                s=100, c=colors[:len(results_df)], zorder=5)
for i, (idx, row) in enumerate(results_df.iterrows()):
    axes[2].annotate(idx, (row['recall'], row['precision']),
                     textcoords='offset points', xytext=(5, 5), fontsize=7)
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision vs. Recall Trade-off')

plt.suptitle('Structured Experiment Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'experiments' / 'experiment_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1800x500 with 3 Axes>

## Loss Curve Comparison Across Experiments

In [ ]:
from scipy.ndimage import uniform_filter1d

def load_results_csv(exp_dir):
    """Load results.csv from an experiment directory."""
    csv = Path(exp_dir) / 'results.csv'
    if not csv.exists():
        return None
    df = pd.read_csv(csv)
    df.columns = df.columns.str.strip()
    return df

exp_dir = ROOT / 'results' / 'experiments'
exp_runs = {
    'E1 imgsz=1280': e1_metrics.get('save_dir'),
    'E2 nano':       e2n_metrics.get('save_dir'),
    'E2 medium':     e2m_metrics.get('save_dir'),
    'E3 lr=0.001':   e3_low_lr.get('save_dir'),
    'E3 cosine LR':  e3_cos_lr.get('save_dir'),
}

fig, axes = plt.subplots(1, 2, figsize=(16, 5))
colors = plt.cm.tab10.colors

for i, (label, save_dir) in enumerate(exp_runs.items()):
    if save_dir is None:
        continue
    df_exp = load_results_csv(save_dir)
    if df_exp is None:
        continue

    val_box = next((c for c in ['val/box_loss', 'val/box_om'] if c in df_exp.columns), None)
    map50   = next((c for c in ['metrics/mAP50(B)', 'metrics/mAP_0.5'] if c in df_exp.columns), None)

    color = colors[i % len(colors)]
    if val_box:
        smoothed = uniform_filter1d(df_exp[val_box].values, size=5)
        axes[0].plot(smoothed, label=label, color=color, linewidth=2)
    if map50:
        axes[1].plot(df_exp[map50].values, label=label, color=color, linewidth=2)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Val Box Loss')
axes[0].set_title('Validation Box Loss  -  All Experiments')
axes[0].legend(fontsize=8)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('mAP50')
axes[1].set_title('mAP50  -  All Experiments')
axes[1].legend(fontsize=8)

plt.suptitle('Loss Curves Across Experiments', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(ROOT / 'results' / 'experiments' / 'loss_curves_comparison.png',
            dpi=150, bbox_inches='tight')
plt.show()

<Figure size 1600x500 with 2 Axes>

: 

## Analysis

Fill in after running the experiments  -  compare each row in the table to the baseline:

### E1  -  Image Resolution (1280)
- **Expected**: Higher resolution captures more detail in small objects.
- **Observed**: (fill in mAP delta vs baseline)
- **Conclusion**: Resolution is / is not the primary bottleneck.

### E2  -  Model Size
- **Expected**: Larger model → higher capacity → better mAP but risk overfitting.
- **Observed**: (fill in)
- **Conclusion**: Model size trade-off on VisDrone.

### E3  -  Learning Rate
- **Expected**: Lower LR = more stable convergence; cosine schedule smoother.
- **Observed**: (fill in)
- **Conclusion**: LR sensitivity analysis.

The best-performing experiment configuration will be carried into **Notebook 04** for iterative improvement.